# 📓 Notebook 1 — Data Loading
### AirSense AI — Intelligent Urban Air Quality Forecasting & Decision Support System

**Purpose of this notebook**

Load the two raw datasets and understand their real structure before touching
anything. No cleaning, no merging, no feature engineering happens here —
that's Notebook 2's job.

**Important — your files come from the Open-Meteo API export format, not a
plain spreadsheet.** Each file has:

| Row | Content |
|---|---|
| 1 | Metadata labels: `latitude, longitude, elevation, utc_offset_seconds, timezone, timezone_abbreviation` |
| 2 | The actual metadata values (station coordinates, GMT offset, timezone) |
| 3 | Blank row |
| 4 | The **real column headers** (`time`, `pm10 (μg/m³)`, etc.) |
| 5+ | The actual hourly data |

So we load this in two passes: once to grab the station metadata, once to
grab the real data table starting at the real header row.

---


## 0. Check required dependencies

Reading `.xlsx` files with pandas requires the `openpyxl` library installed in
**this same environment/virtualenv**. If the next cell errors with
`ModuleNotFoundError: No module named 'openpyxl'`, run this cell once, then
restart the kernel and re-run the notebook from the top.

If you're using a `.venv` in VS Code, make sure the notebook's selected kernel
(top-right of VS Code) points to that same `.venv` — a common cause of this
error is having the `.venv` active in the terminal but a different Python
interpreter selected as the notebook kernel.

In [1]:
import sys
import subprocess

try:
    import openpyxl
    print("openpyxl is already installed — version:", openpyxl.__version__)
except ImportError:
    print("openpyxl not found — installing now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
    print("Installed. If this was your first install, restart the kernel now (Restart button, or Kernel menu) before continuing.")

openpyxl not found — installing now...
Installed. If this was your first install, restart the kernel now (Restart button, or Kernel menu) before continuing.


## 1. Imports

In [2]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

print("pandas version:", pd.__version__)

pandas version: 2.3.3


## 2. Define file paths

Set to your actual project location on disk.

In [3]:
PROJECT_ROOT = Path(r"C:\Users\pc\Desktop\AirSenseAI")

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

AIR_QUALITY_PATH = RAW_DATA_DIR / "air_quality.xlsx"
WEATHER_PATH = RAW_DATA_DIR / "weather.xlsx"

print("Air quality file:", AIR_QUALITY_PATH, "| exists:", AIR_QUALITY_PATH.exists())
print("Weather file:    ", WEATHER_PATH, "| exists:", WEATHER_PATH.exists())

Air quality file: C:\Users\pc\Desktop\AirSenseAI\data\raw\air_quality.xlsx | exists: True
Weather file:     C:\Users\pc\Desktop\AirSenseAI\data\raw\weather.xlsx | exists: True


**If `exists: False` shows up above:** double check that
`air_quality.xlsx` and `weather.xlsx` are actually sitting inside
`C:\Users\pc\Desktop\AirSenseAI\data\raw\`. Fix the path or move the
files before continuing — every cell below depends on this working.

## 3. Extract station metadata

Before loading the actual measurements, we pull out the station metadata
(latitude, longitude, elevation, timezone). This confirms both files come
from the same location (New York City) and tells us the raw timestamps are
in **GMT**, which matters a lot for Notebook 2 when we convert to
`America/New_York`.

In [4]:
def read_openmeteo_metadata(path):
    """Reads the first 2 rows of an Open-Meteo export as station metadata."""
    meta_labels = pd.read_excel(path, header=None, nrows=1).iloc[0].tolist()
    meta_values = pd.read_excel(path, header=None, skiprows=1, nrows=1).iloc[0].tolist()
    return dict(zip(meta_labels, meta_values))

air_quality_meta = read_openmeteo_metadata(AIR_QUALITY_PATH)
weather_meta = read_openmeteo_metadata(WEATHER_PATH)

print("Air Quality station metadata:")
for k, v in air_quality_meta.items():
    print(f"  {k}: {v}")

print()
print("Weather station metadata:")
for k, v in weather_meta.items():
    print(f"  {k}: {v}")

Air Quality station metadata:
  latitude: 40.7
  longitude: -74
  elevation: 51
  utc_offset_seconds: 0
  timezone: GMT
  timezone_abbreviation: GMT

Weather station metadata:
  latitude: 40.7381
  longitude: -74.0425
  elevation: 51
  utc_offset_seconds: 0
  timezone: GMT
  timezone_abbreviation: GMT


**Observation:** _(fill this in after running the cell above)_
Confirm both files report the same (or near-identical) latitude/longitude —
this tells us both datasets represent the same physical location. Also note
the `timezone` field — this is why raw timestamps need to be treated as GMT
before converting to `America/New_York` in Notebook 2.

## 4. Load the Air Quality dataset

The real header row is row **4** (index 3), so we skip the first 3 rows and
use row 3 as the header when reading with pandas.

Expected real columns: `time`, `pm10 (μg/m³)`, `pm2_5 (μg/m³)`,
`ozone (μg/m³)`, `nitrogen_dioxide (μg/m³)`.

`pm2_5 (μg/m³)` is our forecasting **target**.

In [5]:
df_air = pd.read_excel(AIR_QUALITY_PATH, skiprows=3)

print("Shape:", df_air.shape)
df_air.head()

Shape: (17568, 6)


,time,pm10 (μg/m³),pm2_5 (μg/m³),ozone (μg/m³),nitrogen_dioxide (μg/m³),Unnamed: 5
0,2024-01-01 00:00:00,27.8,19.3,0,56.8,NaN
1,2024-01-01 01:00:00,31.3,21.7,0,55.6,NaN
2,2024-01-01 02:00:00,35.5,24.6,0,54.0,NaN
3,2024-01-01 03:00:00,40.0,27.8,0,52.5,NaN
4,2024-01-01 04:00:00,43.9,30.5,0,51.5,NaN


In [6]:
print(list(df_air.columns))
df_air.info()

['time', 'pm10 (μg/m³)', 'pm2_5 (μg/m³)', 'ozone (μg/m³)', 'nitrogen_dioxide (μg/m³)', 'Unnamed: 5']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17568 entries, 0 to 17567
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   time                      17568 non-null  datetime64[ns]
 1   pm10 (μg/m³)              17568 non-null  float64       
 2   pm2_5 (μg/m³)             17568 non-null  float64       
 3   ozone (μg/m³)             17568 non-null  int64         
 4   nitrogen_dioxide (μg/m³)  17568 non-null  float64       
 5   Unnamed: 5                0 non-null      float64       
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 823.6 KB


### 4.1 Drop the trailing empty column

The Open-Meteo export leaves a stray empty trailing column (`Unnamed: 5`) with
zero non-null values. We drop it here — it's a formatting artifact, not real
data.

In [7]:
empty_cols = [c for c in df_air.columns if df_air[c].isna().all()]
print("Fully empty columns found:", empty_cols)

df_air = df_air.drop(columns=empty_cols)
print("Remaining columns:", list(df_air.columns))

Fully empty columns found: ['Unnamed: 5']
Remaining columns: ['time', 'pm10 (μg/m³)', 'pm2_5 (μg/m³)', 'ozone (μg/m³)', 'nitrogen_dioxide (μg/m³)']


## 5. Load the Weather dataset

Same approach — skip the first 3 rows, real header on row 4.

Expected real columns: `time`, `temperature_2m (°C)`,
`relative_humidity_2m (%)`, `wind_speed_10m (km/h)`,
`wind_direction_10m (°)`, `precipitation (mm)`, `cloud_cover (%)`.

⚠️ **Note:** this file does not contain a Surface Pressure column. That's
expected and fine — later notebooks simply won't reference it.

In [8]:
df_weather = pd.read_excel(WEATHER_PATH, skiprows=3)

print("Shape:", df_weather.shape)
df_weather.head()

Shape: (17568, 7)


,time,temperature_2m (°C),relative_humidity_2m (%),wind_speed_10m (km/h),wind_direction_10m (°),precipitation (mm),cloud_cover (%)
0,2024-01-01 00:00:00,3.8,65,5.7,235,0.0,65
1,2024-01-01 01:00:00,3.8,60,6.0,245,0.0,99
2,2024-01-01 02:00:00,3.1,65,6.8,238,0.0,96
3,2024-01-01 03:00:00,2.6,68,8.4,245,0.0,73
4,2024-01-01 04:00:00,1.9,74,6.3,246,0.0,100


In [9]:
print(list(df_weather.columns))
df_weather.info()

['time', 'temperature_2m (°C)', 'relative_humidity_2m (%)', 'wind_speed_10m (km/h)', 'wind_direction_10m (°)', 'precipitation (mm)', 'cloud_cover (%)']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17568 entries, 0 to 17567
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   time                      17568 non-null  datetime64[ns]
 1   temperature_2m (°C)       17568 non-null  float64       
 2   relative_humidity_2m (%)  17568 non-null  int64         
 3   wind_speed_10m (km/h)     17568 non-null  float64       
 4   wind_direction_10m (°)    17568 non-null  int64         
 5   precipitation (mm)        17568 non-null  float64       
 6   cloud_cover (%)           17568 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int64(3)
memory usage: 960.9 KB


In [10]:
expected_weather_cols = {
    "time", "temperature_2m (°C)", "relative_humidity_2m (%)",
    "wind_speed_10m (km/h)", "wind_direction_10m (°)",
    "precipitation (mm)", "cloud_cover (%)"
}
missing = expected_weather_cols - set(df_weather.columns)
extra = set(df_weather.columns) - expected_weather_cols

print("Missing expected columns:", missing if missing else "None")
print("Unexpected extra columns:", extra if extra else "None")

Missing expected columns: None
Unexpected extra columns: None


## 6. Convert `time` to datetime

We explicitly convert both `time` columns to `datetime64`. These are
currently GMT-naive timestamps — we do **not** localize or convert timezone
here, that belongs to Notebook 2. This step just guarantees pandas treats
them as real datetimes, not strings.

In [11]:
df_air['time'] = pd.to_datetime(df_air['time'], errors='coerce')
df_weather['time'] = pd.to_datetime(df_weather['time'], errors='coerce')

print("Air quality — unparseable timestamps:", df_air['time'].isna().sum())
print("Weather      — unparseable timestamps:", df_weather['time'].isna().sum())

Air quality — unparseable timestamps: 0
Weather      — unparseable timestamps: 0


## 7. Check dimensions and time coverage

Confirm row counts and that both files span the expected range:
**2024-01-01 00:00 → 2026-01-01 23:00** (hourly, inclusive), which is
732 days × 24 hours = **17,568 rows** if there are zero gaps.

In [12]:
# 2024-01-01 00:00 through 2026-01-01 23:00 inclusive spans 732 calendar days
# (2024 is a leap year: 366 days, 2025: 365 days, plus the 2026-01-01 day itself = 732)
expected_rows = 732 * 24

print("=== Air Quality ===")
print("Shape:", df_air.shape, "| Expected rows:", expected_rows)
print("Time range:", df_air['time'].min(), "→", df_air['time'].max())

print()
print("=== Weather ===")
print("Shape:", df_weather.shape, "| Expected rows:", expected_rows)
print("Time range:", df_weather['time'].min(), "→", df_weather['time'].max())

=== Air Quality ===
Shape: (17568, 5) | Expected rows: 17568
Time range: 2024-01-01 00:00:00 → 2026-01-01 23:00:00

=== Weather ===
Shape: (17568, 7) | Expected rows: 17568
Time range: 2024-01-01 00:00:00 → 2026-01-01 23:00:00


**Observation:** _(fill this in after running the cell above)_
Note whether the row count matches the expected 17,568 exactly, and whether
both files cover the identical time range. Any mismatch here becomes the
first item on Notebook 2's cleanup list.

## 8. First look at missing values, duplicates, and continuity (observation only)

Still not fixing anything — just documenting what Notebook 2 will need to
address.

In [13]:
print("=== Air Quality — missing values ===")
print(df_air.isna().sum())
print()
print("Duplicate timestamps in Air Quality:", df_air['time'].duplicated().sum())

=== Air Quality — missing values ===
time                        0
pm10 (μg/m³)                0
pm2_5 (μg/m³)               0
ozone (μg/m³)               0
nitrogen_dioxide (μg/m³)    0
dtype: int64

Duplicate timestamps in Air Quality: 0


In [14]:
print("=== Weather — missing values ===")
print(df_weather.isna().sum())
print()
print("Duplicate timestamps in Weather:", df_weather['time'].duplicated().sum())

=== Weather — missing values ===
time                        0
temperature_2m (°C)         0
relative_humidity_2m (%)    0
wind_speed_10m (km/h)       0
wind_direction_10m (°)      0
precipitation (mm)          0
cloud_cover (%)             0
dtype: int64

Duplicate timestamps in Weather: 0


In [15]:
# Check for missing hours (gaps in the hourly sequence)
full_range_air = pd.date_range(df_air['time'].min(), df_air['time'].max(), freq='h')
full_range_weather = pd.date_range(df_weather['time'].min(), df_weather['time'].max(), freq='h')

missing_hours_air = full_range_air.difference(df_air['time'])
missing_hours_weather = full_range_weather.difference(df_weather['time'])

print("Missing hours in Air Quality:", len(missing_hours_air))
print("Missing hours in Weather:", len(missing_hours_weather))

Missing hours in Air Quality: 0
Missing hours in Weather: 0


**Observation:** _(fill this in after running the cells above)_
Note any missing values per column, duplicate timestamps, and missing hours
in the sequence. This becomes the concrete to-do list for Notebook 2's
cleaning step.

## 9. Save lightly-touched copies (checkpoint)

We save the datetime-converted, real-header versions as an intermediate
checkpoint so Notebook 2 can load clean tables directly, without repeating
the row-skipping logic. This is **not** the final processed dataset — that's
`airsense_cube.csv`, produced in Notebook 2.

In [16]:
INTERIM_DIR = PROJECT_ROOT / "data" / "raw"

df_air.to_csv(INTERIM_DIR / "air_quality_parsed.csv", index=False)
df_weather.to_csv(INTERIM_DIR / "weather_parsed.csv", index=False)

print("Saved:")
print("-", INTERIM_DIR / "air_quality_parsed.csv")
print("-", INTERIM_DIR / "weather_parsed.csv")

Saved:
- C:\Users\pc\Desktop\AirSenseAI\data\raw\air_quality_parsed.csv
- C:\Users\pc\Desktop\AirSenseAI\data\raw\weather_parsed.csv


## 10. Summary

At this point we have confirmed:

- ✅ Both raw files load correctly once we account for the Open-Meteo
  metadata header (skip first 3 rows)
- ✅ Real column names identified (`pm10 (μg/m³)`, `pm2_5 (μg/m³)`,
  `ozone (μg/m³)`, `nitrogen_dioxide (μg/m³)`, and the weather equivalents)


- ✅ Dropped one empty artifact column (`Unnamed: 5`) from the air quality file
- ✅ `time` columns are proper `datetime64` objects (still GMT-naive)
- ✅ Row counts checked against the expected 17,568 hourly rows
- ✅ First read on missing values, duplicates, and gaps in the hourly sequence

**Next step → Notebook 2 (ETL & Preprocessing):**
Localize timestamps to GMT then convert to `America/New_York`, merge Air
Quality + Weather on `time`, rename columns to friendly names (`pm2_5` →
`PM2.5`, `temperature_2m` → `Temperature`, etc.), handle any missing
values/gaps, detect outliers, and generate calendar features (Hour, Weekday,
Month, Season, Weekend, ISO Week) before saving the final
`airsense_cube.csv`.